In [7]:
import pandas as pd
import numpy as np
import os

# Sesuaikan nama file dataset mentah kelompok Anda
FILE_MENTAH = 'day.csv'

if os.path.exists(FILE_MENTAH):
    mentah = pd.read_csv(FILE_MENTAH)
    print('Data mentah berhasil dimuat. Ukuran:', mentah.shape)
else:
    print(f'PENTING: Silakan taruh file data mentah Anda di: {FILE_MENTAH}')


Data mentah berhasil dimuat. Ukuran: (731, 16)


In [8]:
def bersihkan_data(df_mentah):
    d = df_mentah.copy()
    jejak = []

    def catat(langkah, terdampak, alasan):
        jejak.append({
            'langkah': langkah,
            'baris_terdampak': int(terdampak),
            'alasan': alasan
        })
        print(f'  [LOG] {langkah}: {int(terdampak)} baris')

    # -------------------------------------------------------------------------
    kolom_teks = [c for c in d.columns if d[c].dtype == object or str(d[c].dtype) == 'str']
    n_spasi = 0
    for kol in kolom_teks:
        s = d[kol].astype('string')
        n_spasi += int((s != s.str.strip()).sum())
        d[kol] = s.str.strip()
    catat('Buang spasi awal/akhir teks', n_spasi, 'Menghindari inkonsistensi pencarian teks akibat spasi tersembunyi')

    # -------------------------------------------------------------------------
    n_persis = int(d.duplicated().sum())
    d = d.drop_duplicates()
    catat('Hapus duplikat persis', n_persis, 'Menghapus baris yang seluruh kolomnya identik')


    df_bersih = d.reset_index(drop=True)
    df_log = pd.DataFrame(jejak)
    df_log.index = range(1, len(df_log) + 1)

    return df_bersih, df_log

In [10]:
def bersihkan_bike_sharing(df_input):
    """
    Fungsi pipeline untuk membersihkan Bike Sharing Dataset (day.csv)
    mengikuti struktur dan urutan logis Modul P05.
    """
    x = df_input.copy()
    log = []

    def catat(langkah, n, alasan):
        log.append({
            'langkah': langkah,
            'baris_terdampak': int(n),
            'alasan': alasan
        })

    # 1. Tipe Data: Konversi dteday ke datetime
    x['dteday'] = pd.to_datetime(x['dteday'], format='%Y-%m-%d', errors='coerce')
    catat('Konversi dteday ke datetime', len(x),
          'Memastikan kolom tanggal memiliki tipe datetime yang valid untuk analisis deret waktu')

    # 2. Buang Spasi pada kolom teks (jika ada)
    kolom_teks = [col for col in x.columns if x[col].dtype == object or str(x[col].dtype) == 'str']
    n_spasi = 0
    for kol in kolom_teks:
        s = x[kol].astype('string')
        n_spasi += int((s != s.str.strip()).sum())
        x[kol] = s.str.strip()
    catat('Buang spasi awal dan akhir kolom teks', n_spasi,
          'Memastikan tidak ada spasi tersembunyi pada kolom teks')

    # 3. Penyeragaman & Pemetaan Kategori Deskriptif
    peta_season = {1: 'Spring', 2: 'Summer', 3: 'Fall', 4: 'Winter'}
    x['season_label'] = x['season'].map(peta_season)
    catat('Pemetaan kategori season ke label', len(x),
          'Mengubah kode numerik 1-4 menjadi label deskriptif untuk keterbacaan EDA')

    peta_weather = {1: 'Clear/Few clouds', 2: 'Mist/Cloudy', 3: 'Light Snow/Rain', 4: 'Heavy Rain/Ice Pallets'}
    x['weathersit_label'] = x['weathersit'].map(peta_weather)
    catat('Pemetaan kategori weathersit ke label', len(x),
          'Mengubah kode cuaca numerik menjadi kategori deskriptif')

    # 4. Tangani Duplikat
    n_dup_persis = int(x.duplicated().sum())
    x = x.drop_duplicates()
    catat('Hapus duplikat persis', n_dup_persis,
          'Seluruh kolom identik; memastikan satu baris tersimpan per hari')

    n_dup_key = int(x['instant'].duplicated().sum())
    x = x.drop_duplicates(subset=['instant'], keep='first')
    catat('Hapus duplikat pada kunci utama instant', n_dup_key,
          'Memastikan instant sebagai primary key bernilai unik')

    # 5. Tangani Nilai Mustahil / Abnormal (Kelembapan & Windspeed)
    hum_bad = (x['hum'] <= 0) | (x['hum'] > 1)
    n_hum_bad = int(hum_bad.sum())
    x.loc[hum_bad, 'hum'] = np.nan
    catat('Kelembapan (hum) <= 0 atau > 1 menjadi NaN', n_hum_bad,
          'Kelembapan <= 0% atau > 100% merupakan kesalahan pengukuran sensor')

    wind_bad = (x['windspeed'] < 0)
    n_wind_bad = int(wind_bad.sum())
    x.loc[wind_bad, 'windspeed'] = np.nan
    catat('Windspeed negatif menjadi NaN', n_wind_bad,
          'Kecepatan angin tidak mungkin bernilai negatif')

    # 6. Tandai Konsistensi Total Sewa (casual + registered == cnt)
    x['cnt_seharusnya'] = x['casual'] + x['registered']
    x['total_mencurigakan'] = x['cnt'] != x['cnt_seharusnya']
    n_curiga = int(x['total_mencurigakan'].sum())
    catat('Tandai ketidakcocokan total sewaan (cnt)', n_curiga,
          'Ditandai dengan kolom boolean, tidak ditimpa langsung agar dapat diverifikasi pada tahap analisis')

    # 7. Kolom Turunan Tanggal
    x['tahun_asli'] = x['dteday'].dt.year
    x['bulan_nama'] = x['dteday'].dt.month_name()
    x['hari_nama'] = x['dteday'].dt.day_name()
    catat('Buat kolom turunan tanggal', len(x),
          'Mengekstrak tahun, nama bulan, dan nama hari untuk memfasilitasi visualisasi dan agregasi')

    return x.reset_index(drop=True), pd.DataFrame(log)

print("Fungsi pipeline `bersihkan_bike_sharing` siap dijalankan.")

Fungsi pipeline `bersihkan_bike_sharing` siap dijalankan.


In [12]:
df_bersih, df_log = bersihkan_bike_sharing(mentah)

print("Proses Pembersihan Selesai!")
print(f"Data Mentah : {mentah.shape}")
print(f"Data Bersih : {df_bersih.shape}")

Proses Pembersihan Selesai!
Data Mentah : (731, 16)
Data Bersih : (731, 23)


In [13]:
df_log.index = range(1, len(df_log) + 1)
df_log

,langkah,baris_terdampak,alasan
1,Konversi dteday ke datetime,731,Memastikan kolom tanggal memiliki tipe datetim...
2,Buang spasi awal dan akhir kolom teks,0,Memastikan tidak ada spasi tersembunyi pada ko...
3,Pemetaan kategori season ke label,731,Mengubah kode numerik 1-4 menjadi label deskri...
4,Pemetaan kategori weathersit ke label,731,Mengubah kode cuaca numerik menjadi kategori d...
5,Hapus duplikat persis,0,Seluruh kolom identik; memastikan satu baris t...
6,Hapus duplikat pada kunci utama instant,0,Memastikan instant sebagai primary key bernila...
7,Kelembapan (hum) <= 0 atau > 1 menjadi NaN,1,Kelembapan <= 0% atau > 100% merupakan kesalah...
8,Windspeed negatif menjadi NaN,0,Kecepatan angin tidak mungkin bernilai negatif
9,Tandai ketidakcocokan total sewaan (cnt),0,"Ditandai dengan kolom boolean, tidak ditimpa l..."
10,Buat kolom turunan tanggal,731,"Mengekstrak tahun, nama bulan, dan nama hari u..."


In [16]:
import os
from google.colab import drive

# 1. Mount Google Drive terlebih dahulu
drive.mount('/content/drive')

# 2. Ganti 'kelompok_eda-p5' dengan nama folder yang sesuai di Drive Anda
path_folder = '/content/drive/MyDrive/EDA/kelompok_eda-p5'

if os.path.exists(path_folder):
    os.chdir(path_folder)
    print("Direktori aktif saat ini:", os.getcwd())
else:
    print(f"PENTING: Folder '{path_folder}' tidak ditemukan. Silakan periksa kembali nama folder Anda di Google Drive.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Direktori aktif saat ini: /content/drive/MyDrive/EDA/kelompok_eda-p5


In [18]:
import os

# 1. Tentukan path direktori tempat notebook berjalan
# Menggunakan './' agar langsung menyimpan di folder aktif saat ini
DIR_PROCESSED = 'data/processed'
DIR_REPORTS = 'reports'

os.makedirs(DIR_PROCESSED, exist_ok=True)
os.makedirs(DIR_REPORTS, exist_ok=True)

# 2. Simpan dataset bersih ke folder data/processed/
PATH_PROCESSED = os.path.join(DIR_PROCESSED, 'day_clean.csv')
df_bersih.to_csv(PATH_PROCESSED, index=False)

# 3. Buat file cleaning_log.md di folder aktif (root proyek)
log_lines = [
    "# Cleaning Log — Bike Sharing Dataset\n",
    f"Dataset: `day.csv`  ",
    f"Baris Mentah: {len(mentah)} | Baris Bersih: {len(df_bersih)}  \n",
    "| # | Langkah | Baris Terdampak | Alasan / Justifikasi |",
    "|---|---|---|---|"
]

for i, row in df_log.iterrows():
    log_lines.append(f"| {i} | {row['langkah']} | {row['baris_terdampak']} | {row['alasan']} |")

PATH_LOG = 'cleaning_log.md'
with open(PATH_LOG, 'w', encoding='utf-8') as f:
    f.write('\n'.join(log_lines))

# CETAK OUTPUT SECARA EKSPLISIT MEGGUNAKAN PRINT
print("=" * 50)
print("STATUS PENYIMPANAN:")
print(f"✓ Data bersih tersimpan di : {os.path.abspath(PATH_PROCESSED)}")
print(f"✓ Cleaning log tersimpan di: {os.path.abspath(PATH_LOG)}")
print("=" * 50)

STATUS PENYIMPANAN:
✓ Data bersih tersimpan di : /content/drive/MyDrive/EDA/kelompok_eda-p5/data/processed/day_clean.csv
✓ Cleaning log tersimpan di: /content/drive/MyDrive/EDA/kelompok_eda-p5/cleaning_log.md


In [19]:
# Verifikasi hasil pembersihan
print("=== RINGKASAN VERIFIKASI DATA ===")
print("Jumlah Baris Mentah        :", len(mentah))
print("Jumlah Baris Bersih        :", len(df_bersih))
print("Null pada kelembapan (hum) :", df_bersih['hum'].isna().sum())
print("Tipe data dteday           :", df_bersih['dteday'].dtype)
print("Total sewa tidak cocok     :", df_bersih['total_mencurigakan'].sum())

=== RINGKASAN VERIFIKASI DATA ===
Jumlah Baris Mentah        : 731
Jumlah Baris Bersih        : 731
Null pada kelembapan (hum) : 1
Tipe data dteday           : datetime64[ns]
Total sewa tidak cocok     : 0


In [20]:
if 'mentah' in globals():
    bersih, log = bersihkan_data(mentah)

    # Simpan dataset bersih
    os.makedirs('data/processed', exist_ok=True)
    bersih.to_csv('day.csv', index=False)
    print('Data bersih sukses disimpan di day.csv')

    # Buat file cleaning_log.md secara otomatis
    baris_log = [
        '# Cleaning Log Tim', '',
        f'Dataset: {FILE_MENTAH}  ',
        f'Baris mentah: {len(mentah)} -> baris bersih: {len(bersih)}  ', '',
        '| # | Langkah | Baris terdampak | Alasan |',
        '|---|---|---|---|'
    ]
    for idx, r in log.iterrows():
        baris_log.append(f"| {idx} | {r['langkah']} | {r['baris_terdampak']} | {r['alasan']} |")

    with open('cleaning_log.md', 'w', encoding='utf-8') as f:
        f.write('\n'.join(baris_log))
    print('cleaning_log.md sukses diperbarui!')


  [LOG] Buang spasi awal/akhir teks: 0 baris
  [LOG] Hapus duplikat persis: 0 baris
Data bersih sukses disimpan di day.csv
cleaning_log.md sukses diperbarui!
